# Monte Carlo Methods: Modular Generalized Policy Iteration (GPI)

This notebook provides a comprehensive pedagogical demonstration of Monte Carlo (MC) methods. We break the algorithm into modular components to illustrate the inputs and outputs of each phase and compare different exploration strategies.

## 1. Visual Overview: How MC Works

<svg width="800" height="350" viewBox="0 0 800 350" xmlns="http://www.w3.org/2000/svg">
  <style>
    .box { fill: #ffffff; stroke: #343a40; stroke-width: 2; }
    .title { font-family: sans-serif; font-size: 16px; font-weight: bold; fill: #212529; }
    .desc { font-family: sans-serif; font-size: 12px; fill: #495057; }
    .arrow { fill: none; stroke: #007bff; stroke-width: 2; marker-end: url(#arrowhead); }
    .highlight { fill: #f1f8ff; stroke: #007bff; }
    .step-num { font-family: sans-serif; font-size: 20px; font-weight: bold; fill: #dee2e6; }
  </style>
  <defs>
    <marker id="arrowhead" markerWidth="10" markerHeight="7" refX="0" refY="3.5" orient="auto">
      <polygon points="0 0, 10 3.5, 0 7" fill="#007bff" />
    </marker>
  </defs>
  <!-- Step 1 -->
  <rect x="10" y="50" width="180" height="120" class="box highlight" rx="8" />
  <text x="20" y="40" class="step-num">01</text>
  <text x="100" y="80" class="title" text-anchor="middle">Experience</text>
  <text x="100" y="105" class="desc" text-anchor="middle">Pick a RANDOM state & action</text>
  <text x="100" y="120" class="desc" text-anchor="middle">(Exploring Start). Follow the</text>
  <text x="100" y="135" class="desc" text-anchor="middle">current Policy to the Goal.</text>
  <!-- Step 2 -->
  <rect x="210" y="50" width="180" height="120" class="box" rx="8" />
  <text x="220" y="40" class="step-num">02</text>
  <text x="300" y="80" class="title" text-anchor="middle">Backtracking</text>
  <text x="300" y="105" class="desc" text-anchor="middle">The game ends. Look back</text>
  <text x="300" y="120" class="desc" text-anchor="middle">at the path and calculate</text>
  <text x="300" y="135" class="desc" text-anchor="middle">the total Return (G).</text>
  <!-- Step 3 -->
  <rect x="410" y="50" width="180" height="120" class="box" rx="8" />
  <text x="420" y="40" class="step-num">03</text>
  <text x="500" y="80" class="title" text-anchor="middle">Evaluation</text>
  <text x="500" y="105" class="desc" text-anchor="middle">Update scorecard N(s,a).</text>
  <text x="500" y="120" class="desc" text-anchor="middle">Find the NEW average Q(s,a)</text>
  <text x="500" y="135" class="desc" text-anchor="middle">using the observed return G.</text>
  <!-- Step 4 -->
  <rect x="610" y="50" width="180" height="120" class="box" rx="8" />
  <text x="620" y="40" class="step-num">04</text>
  <text x="700" y="80" class="title" text-anchor="middle">Improvement</text>
  <text x="700" y="105" class="desc" text-anchor="middle">Update the map. Pick the</text>
  <text x="700" y="120" class="desc" text-anchor="middle">action with the highest</text>
  <text x="700" y="135" class="desc" text-anchor="middle">historical average (Greedy).</text>
  <!-- Arrows -->
  <path d="M 190 110 L 205 110" class="arrow" />
  <path d="M 390 110 L 405 110" class="arrow" />
  <path d="M 590 110 L 605 110" class="arrow" />
  <path d="M 700 170 L 700 220 L 100 220 L 100 180" class="arrow" />
  <text x="400" y="250" class="title" text-anchor="middle" style="fill:#007bff">The GPI Cycle (Repeat Thousands of Times)</text>
</svg>

## 2. Mathematical Foundations

### The Return ($G_t$)
Total discounted reward from time step $t$ until the end of the episode:
$$G_t \doteq R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots = \sum_{k=0}^{T-t-1} \gamma^k R_{t+k+1}$$

### Monte Carlo Evaluation ($Q$)
We estimate the action-value function $q_\pi(s, a)$ by averaging returns across many independent episodes. We track the number of visits $N(s, a)$ to calculate the empirical mean:
$$Q(s, a) = \frac{1}{N(s, a)} \sum_{i=1}^{N(s, a)} G_i(s, a)$$

### Monte Carlo Control ($\pi$)
In Generalized Policy Iteration (GPI), we improve the policy by making it greedy with respect to our current estimates:
$$\pi(s) \doteq \arg\max_a Q(s, a)$$

## 3. Environment Setup: 3x3 GridWorld

<svg width="400" height="400" viewBox="0 0 400 400" xmlns="http://www.w3.org/2000/svg">
  <style>
    .cell { fill: #f8f9fa; stroke: #343a40; stroke-width: 2; }
    .text { font-family: sans-serif; font-size: 16px; fill: #343a40; text-anchor: middle; }
    .reward { font-size: 12px; fill: #6c757d; font-style: italic; }
    .goal { fill: #d4edda; }
    .goal-text { fill: #155724; font-weight: bold; }
    .arrow { fill: none; stroke: #007bff; stroke-width: 1.5; marker-end: url(#arrowhead); opacity: 0.6; }
  </style>
  <defs>
    <marker id="arrowhead" markerWidth="10" markerHeight="7" refX="0" refY="3.5" orient="auto">
      <polygon points="0 0, 10 3.5, 0 7" fill="#007bff" />
    </marker>
  </defs>
  <rect x="50" y="50" width="100" height="100" class="cell" />
  <text x="100" y="80" class="text">State 0</text>
  <text x="100" y="100" class="text reward">R: -1</text>
  <rect x="150" y="50" width="100" height="100" class="cell" />
  <text x="200" y="80" class="text">State 1</text>
  <text x="200" y="100" class="text reward">R: -1</text>
  <rect x="250" y="50" width="100" height="100" class="cell" />
  <text x="300" y="80" class="text">State 2</text>
  <text x="300" y="100" class="text reward">R: -1</text>
  <rect x="50" y="150" width="100" height="100" class="cell" />
  <text x="100" y="180" class="text">State 3</text>
  <text x="100" y="200" class="text reward">R: -1</text>
  <rect x="150" y="150" width="100" height="100" class="cell" />
  <text x="200" y="180" class="text">State 4</text>
  <text x="200" y="200" class="text reward">R: -1</text>
  <rect x="250" y="150" width="100" height="100" class="cell" />
  <text x="300" y="180" class="text">State 5</text>
  <text x="300" y="200" class="text reward">R: -1</text>
  <rect x="50" y="250" width="100" height="100" class="cell" />
  <text x="100" y="280" class="text">State 6</text>
  <text x="100" y="300" class="text reward">R: -1</text>
  <rect x="150" y="250" width="100" height="100" class="cell" />
  <text x="200" y="280" class="text">State 7</text>
  <text x="200" y="300" class="text reward">R: -1</text>
  <rect x="250" y="250" width="100" height="100" class="cell goal" />
  <text x="300" y="280" class="text goal-text">State 8</text>
  <text x="300" y="300" class="text reward goal-text">R: +10</text>
  <path d="M 200 160 L 200 135" class="arrow" />
  <path d="M 200 220 L 200 245" class="arrow" />
  <path d="M 170 190 L 145 190" class="arrow" />
  <path d="M 230 190 L 255 190" class="arrow" />
</svg>

In [ ]:
import numpy as np
import random
from collections import defaultdict

class SimpleGridWorld:
    def __init__(self):
        self.size = 3
        self.goal = 8
    
    def reset(self, start_state=0):
        # For Exploring Starts, we allow forcing the environment to any state
        self.state = start_state
        return self.state
    
    def step(self, action):
        row, col = divmod(self.state, self.size)
        if action == 0: row = max(0, row - 1)    # Up
        elif action == 1: row = min(self.size - 1, row + 1) # Down
        elif action == 2: col = max(0, col - 1)  # Left
        elif action == 3: col = min(self.size - 1, col + 1) # Right
        
        self.state = row * self.size + col
        reward = 10 if self.state == self.goal else -1
        done = (self.state == self.goal)
        return self.state, reward, done

def update_q_values(episode, Q, returns_sum, N, gamma=0.9):
    G = 0
    visited_sa = set()
    for t in range(len(episode) - 1, -1, -1):
        s, a, r = episode[t]
        G = gamma * G + r
        # First-visit MC logic
        if (s, a) not in visited_sa:
            visited_sa.add((s, a))
            returns_sum[s][a] += G
            N[s][a] += 1
            Q[s][a] = returns_sum[s][a] / N[s][a]

## 4. Implementation Details

### The "Initial Guess"
We start with an **arbitrary policy** (e.g., "always go Right"). 
- The diversity in our 1000s of episodes comes from either **Exploring Starts** (Method A) or **Epsilon-Greedy Actions** (Method B).

### Why do we need a policy during data collection?
The policy acts as a **guide**. After the first action, the agent follows its plan to ensure it reaches the goal. Without this, the agent might wander forever and never learn anything.

## 5. Method A: Monte Carlo ES (Exploring Starts)

**Logic**: Uses a purely **Deterministic** (Greedy) policy, but forces exploration by starting every episode in a **RANDOM** state and taking a **RANDOM** first action.

In [ ]:
def run_mc_es(num_iterations=20):
    env = SimpleGridWorld()
    Q = defaultdict(lambda: np.zeros(4))
    returns_sum = defaultdict(lambda: np.zeros(4))
    N = defaultdict(lambda: np.zeros(4))
    policy = defaultdict(lambda: 3) # Initial guess: All Right

    print("--- Running Monte Carlo ES (Random Starts) ---")
    for i in range(num_iterations):
        # EXPLORING START: Force random starting state
        s0 = random.choice(range(8))
        a0 = random.choice(range(4))
        
        episode = []
        state = env.reset(s0)
        
        # Step 1: Execute random first action
        next_state, reward, done = env.step(a0)
        episode.append((state, a0, reward))
        state = next_state
        
        # Step 2: Follow Greedy Policy until terminal
        while not done:
            action = policy[state]
            next_state, reward, done = env.step(action)
            episode.append((state, action, reward))
            state = next_state
            
        update_q_values(episode, Q, returns_sum, N)
        
        # Improvement: Make policy greedy w.r.t Q
        for s in range(8):
            policy[s] = np.argmax(Q[s])
            
    print("Final Policy Grid (ES):")
    print(np.array([policy[s] for s in range(9)]).reshape(3,3))

run_mc_es()

## 6. Method B: On-Policy $\epsilon$-Greedy MC

**Logic**: Starts from a **FIXED** state (State 0) every time, but forces exploration by taking **RANDOM ACTIONS** with probability $\epsilon$ throughout the episode.

In [ ]:
def run_mc_epsilon_greedy(num_iterations=200, epsilon=0.2):
    env = SimpleGridWorld()
    Q = defaultdict(lambda: np.zeros(4))
    returns_sum = defaultdict(lambda: np.zeros(4))
    N = defaultdict(lambda: np.zeros(4))

    print(f"--- Running Epsilon-Greedy MC (Fixed Start, eps={epsilon}) ---")
    for i in range(num_iterations):
        # FIXED START
        state = env.reset(0) 
        episode = []
        done = False
        
        while not done:
            # EPSILON-GREEDY SELECTION
            if random.random() < epsilon:
                action = random.choice(range(4))
            else:
                action = np.argmax(Q[state])
                
            next_state, reward, done = env.step(action)
            episode.append((state, action, reward))
            state = next_state
            if len(episode) > 100: break
            
        update_q_values(episode, Q, returns_sum, N)
        
    print("Final Policy Grid (Epsilon-Greedy):")
    final_policy = [np.argmax(Q[s]) for s in range(9)]
    print(np.array(final_policy).reshape(3,3))

run_mc_epsilon_greedy()

## 7. Key Takeaways

1. **Monte Carlo ES**: Learns fast by sampling the entire world immediately. Best for deterministic policies when you can "reset" the agent anywhere.
2. **$\epsilon$-Greedy**: More realistic. The agent starts at the beginning and must find its own way by occasionally being random. Convergence is slower but more robust for real-world tasks.